In [ ]:
from lsst.ts.m1m3.utils.thermocouples import ThermocoupleAnalysis
from astropy.time import Time, TimeDelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib import cm

from lsst.ts.xml.tables.m1m3 import ThermocoupleTable
import numpy as np 

%matplotlib widget 

# make EFD client
from lsst_efd_client import EfdClient

In [ ]:
efd = EfdClient('usdf_efd')

# stability test #2 after nozzle installations
time_start = Time('2026-04-14 08:00:00')
time_end = Time('2026-04-14 20:00:00')

#stability test #1 before nozzle installations, three steps
#time_start = Time('2026-03-04 18:30:00')
#time_end = Time('2026-03-05 12:30:00')

thermocouples = ThermocoupleAnalysis(efd)
await thermocouples.load(time_start, time_end)

In [ ]:
standard_gradients = thermocouples.xyz_r_gradients

In [ ]:
standard_bulk_stats = thermocouples.bulk_glass_temperature_metrics

***
Time to start exploring some radius cuts

Some radii for reference:
- M3 outer radius: 3.4
- M3 middle ring: 1.83
- M3 inner rings: 0.58

In [ ]:
m3_radius_limit =2.5

In [ ]:
def count_standard_sensors_within_radius(analysis, radius_limit, return_names=True):
    """
    Count standard thermocouples within a radius cut.

    Parameters
    ----------
    analysis : ThermocoupleAnalysis
        Needed for nonstandard thermocouple list.
    radius_limit : float
    return_names : bool

    Returns
    -------
    count : int
    names : list[str]
    """

    nonstandard = {tc.name for tc in analysis.nonstandard_thermocouples}

    names = []
    for tc in ThermocoupleTable:
        if tc.name in nonstandard:
            continue

        r = np.hypot(tc.x_position, tc.y_position)

        if r <= radius_limit:
            names.append(tc.name)

    count = len(names)

    print(f"Radius ≤ {radius_limit:.2f} → {count} standard thermocouples")

    if return_names:
        return count, names
    else:
        return count

In [ ]:
def plot_standard_count_vs_radius(analysis, r_max=None, n_steps=100, vertical_lines=None,
    line_kwargs=None,
):
    """
    Plot number of standard thermocouples vs radius cut.
    """

    if line_kwargs is None:
        line_kwargs = {}

    # get all radii (standard only)
    nonstandard = {tc.name for tc in analysis.nonstandard_thermocouples}

    radii = [
        np.hypot(tc.x_position, tc.y_position)
        for tc in ThermocoupleTable
        if tc.name not in nonstandard
    ]

    radii = np.array(radii)

    if r_max is None:
        r_max = radii.max()

    r_grid = np.linspace(0, r_max, n_steps)

    counts = []
    for r in r_grid:
        count = np.sum(radii <= r)
        counts.append(count)

    plt.figure()
    plt.plot(r_grid, counts)

    # vertical lines
    if vertical_lines is not None:
        for r in vertical_lines:
            plt.axvline(
                r,
                linestyle="--",
                color="red",
                **line_kwargs,
            )
    
    plt.xlabel("Radius")
    plt.ylabel("Number of standard thermocouples")
    plt.title("Thermocouple count vs radius cut")
    plt.grid(True)

    return r_grid, counts

In [ ]:
plot_standard_count_vs_radius(thermocouples, r_max=None, n_steps=100, vertical_lines=[m3_radius_limit])

In [ ]:
count_standard_sensors_within_radius(thermocouples, radius_limit=m3_radius_limit)

In [ ]:
count_standard_sensors_within_radius(thermocouples, radius_limit=7.5)

In [ ]:
inner_gradients = thermocouples.calculate_gradients_xyz_r(radius_limit=m3_radius_limit)

inner_bulk_stats = thermocouples.compute_temp_stats_and_rate(
    data=thermocouples.all_thermocouples_dataframe,
    radius_limit=3.0,
)

inner_vertical_stats = thermocouples.compute_temp_stats_and_rate(
    data=thermocouples.vertical_cell_gradient_dataframe,
    radius_limit=3.0,
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# --- Palettes: same hue family within groups, distinct shades per line ---
blues   = plt.get_cmap('Blues')
oranges = plt.get_cmap('Oranges')

# Line colors (distinct, but clearly grouped)
c_x = blues(0.75)     # darker blue
c_y = blues(0.55)     # lighter blue
c_r = oranges(0.75)   # darker orange
c_z = oranges(0.55)   # lighter orange

# Band colors (very light tint of the group hue)
band_xy = blues(0.25)
band_rz = oranges(0.25)

# --- Shaded operational bands (put behind data) ---
ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.15, zorder=0)
ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.20, zorder=0)
ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)

# --- Data lines: distinct per series with styles/markers ---
ln_x, = ax.plot(
    standard_gradients.x_gradient * 8.4,
    label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
    marker="o", markersize=3, markevery=50, zorder=3
)
ln_y, = ax.plot(
    standard_gradients.y_gradient * 8.4,
    label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
    marker="s", markersize=3, markevery=50, zorder=3
)
ln_r, = ax.plot(
    standard_gradients.radial_gradient * 3.7,
    label="Radial (×3.7)", color=c_r, linewidth=2.0, linestyle="-",
    marker="^", markersize=3, markevery=50, zorder=4
)
ln_z, = ax.plot(
    standard_gradients.z_gradient,
    label="Z", color=c_z, linewidth=2.0, linestyle="--",
    marker="D", markersize=3, markevery=50, zorder=4
)

# --- Time formatting (if index is datetime) ---
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

# --- Colored labels on the right, matching each band hue ---
ax.annotate("X/Y operational limit ±0.4",
            xy=(1, 0.25), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=blues(0.6), fontsize=9)
ax.annotate("Radial/Z operational limit ±0.1",
            xy=(1, 0.0), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=oranges(0.6), fontsize=9)

# --- Legends: one for data lines, one for bands (with matching colors) ---
data_leg = ax.legend(handles=[ln_x, ln_y, ln_r, ln_z],
                     loc="upper left", frameon=True)
ax.add_artist(data_leg)

band_handles = [
    Patch(facecolor=band_xy, alpha=0.25, label="X/Y limit band (±0.4)"),
    Patch(facecolor=band_rz, alpha=0.25, label="Radial/Z limit band (±0.1)"),
]
ax.legend(handles=band_handles, title="Operational Limits",
          loc="lower left", frameon=True)

# --- Labels, grid, cosmetics ---

# --- Gather all y-data across plotted series ---
ydata = np.concatenate([
    ln_x.get_ydata(),
    ln_y.get_ydata(),
    ln_r.get_ydata(),
    ln_z.get_ydata(),
])

# --- Compute the extrema safely ---
ymin = np.nanmin(ydata)
ymax = np.nanmax(ydata)

# --- Apply conditional limits ---
lower = ymin if ymin < -0.8 else -0.8
upper = ymax if ymax >  0.8 else  0.8
ax.set_ylim(lower, upper)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Thermal Gradient (deg C)")
ax.set_title("M1M3 Thermal Gradients Across the Mirror")
ax.grid(True, linestyle=":", alpha=0.6)
# Show minutes in the x-axis labels (and full date so it’s unambiguous)
locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# --- Palettes: same hue family within groups, distinct shades per line ---
blues   = plt.get_cmap('Blues')
oranges = plt.get_cmap('Oranges')

# Line colors (distinct, but clearly grouped)
c_x = blues(0.75)     # darker blue
c_y = blues(0.55)     # lighter blue
c_r = oranges(0.75)   # darker orange
c_z = oranges(0.55)   # lighter orange

# Band colors (very light tint of the group hue)
band_xy = blues(0.25)
band_rz = oranges(0.25)

# --- Shaded operational bands (put behind data) ---
ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.15, zorder=0)
ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.20, zorder=0)
ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)

# --- Data lines: distinct per series with styles/markers ---
ln_x, = ax.plot(
    inner_gradients.x_gradient * 8.4,
    label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
    marker="o", markersize=3, markevery=50, zorder=3
)
ln_y, = ax.plot(
    inner_gradients.y_gradient * 8.4,
    label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
    marker="s", markersize=3, markevery=50, zorder=3
)
ln_r, = ax.plot(
    inner_gradients.radial_gradient * 3.7,
    label="Radial (×3.7)", color=c_r, linewidth=2.0, linestyle="-",
    marker="^", markersize=3, markevery=50, zorder=4
)
ln_z, = ax.plot(
    inner_gradients.z_gradient,
    label="Z", color=c_z, linewidth=2.0, linestyle="--",
    marker="D", markersize=3, markevery=50, zorder=4
)

# --- Time formatting (if index is datetime) ---
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

# --- Colored labels on the right, matching each band hue ---
ax.annotate("X/Y operational limit ±0.4",
            xy=(1, 0.25), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=blues(0.6), fontsize=9)
ax.annotate("Radial/Z operational limit ±0.1",
            xy=(1, 0.0), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=oranges(0.6), fontsize=9)

# --- Legends: one for data lines, one for bands (with matching colors) ---
data_leg = ax.legend(handles=[ln_x, ln_y, ln_r, ln_z],
                     loc="upper left", frameon=True)
ax.add_artist(data_leg)

band_handles = [
    Patch(facecolor=band_xy, alpha=0.25, label="X/Y limit band (±0.4)"),
    Patch(facecolor=band_rz, alpha=0.25, label="Radial/Z limit band (±0.1)"),
]
ax.legend(handles=band_handles, title="Operational Limits",
          loc="lower left", frameon=True)

# --- Labels, grid, cosmetics ---

# --- Gather all y-data across plotted series ---
ydata = np.concatenate([
    ln_x.get_ydata(),
    ln_y.get_ydata(),
    ln_r.get_ydata(),
    ln_z.get_ydata(),
])

# --- Compute the extrema safely ---
ymin = np.nanmin(ydata)
ymax = np.nanmax(ydata)

# --- Apply conditional limits ---
lower = ymin if ymin < -0.8 else -0.8
upper = ymax if ymax >  0.8 else  0.8
ax.set_ylim(lower, upper)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Thermal Gradient (deg C)")
ax.set_title("M1M3 Thermal Gradients Across Inner M3")
ax.grid(True, linestyle=":", alpha=0.6)
# Show minutes in the x-axis labels (and full date so it’s unambiguous)
locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
inner_gradients = thermocouples.calculate_gradients_xyz_r(radius_limit=m3_radius_limit)
standard_gradients = thermocouples.xyz_r_gradients

standard_bulk_stats = thermocouples.bulk_glass_temperature_metrics
inner_bulk_stats = thermocouples.compute_temp_stats_and_rate(
    data=thermocouples.all_thermocouples_dataframe,
    radius_limit=3.0,
)

In [ ]:
def _infer_ylabel(col):
    if "x_gradient" in col:
        return "X Gradient (deg C/m)"
    elif "y_gradient" in col:
        return "Y Gradient (deg C/m)"
    elif "z_gradient" in col:
        return "Z Gradient (deg C/m)"
    elif "radial_gradient" in col:
        return "Radial Gradient (deg C/m)"
    elif "rate" in col:
        return "Rate of Change (deg C/min)"
    elif "mean_temp" in col:
        return "Mean Temperature (deg C)"
    elif "std_temp" in col:
        return "Temperature Std Dev (deg C)"
    elif "range_temp" in col:
        return "Temperature Range (deg C)"
    else:
        return col


def plot_full_vs_inner_timeseries_with_diff(
    full,
    inner,
    columns,
):
    common_index = full.index.intersection(inner.index)

    for col in columns:
        fig, (ax, ax_diff) = plt.subplots(
            2, 1,
            figsize=(11, 6),
            sharex=True,
            gridspec_kw={"height_ratios": [3, 1]},
        )

        y_full = full.loc[common_index, col]
        y_inner = inner.loc[common_index, col]
        y_diff = y_inner - y_full

        # --- TOP PANEL ---
        ax.plot(
            common_index,
            y_full,
            label="Full M1M3",
            color="0.25",          # dark gray
            linewidth=2.2,
        )

        ax.plot(
            common_index,
            y_inner,
            label="Inner M3",
            color="#1f77b4",      # clear blue
            linewidth=2.2,
            alpha=0.8,
        )

        ax.set_ylabel(_infer_ylabel(col))
        ax.set_title(f"{col}: Full vs Inner")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend()

        # --- LOWER PANEL ---
        ax_diff.plot(
            common_index,
            y_diff,
            label="Inner - Full",
            color="black",
            linewidth=1.5,
        )

        ax_diff.axhline(0, color="0.5", linestyle="--", linewidth=1)

        ax_diff.set_xlabel("Time (UTC)")
        ax_diff.set_ylabel("Inner - Full")
        ax_diff.grid(True, linestyle=":", alpha=0.6)

        # --- TIME AXIS ---
        locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
        ax_diff.xaxis.set_major_locator(locator)
        ax_diff.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))

        plt.setp(ax_diff.get_xticklabels(), rotation=30, ha="right")

        plt.tight_layout()
        plt.show()

In [ ]:
plot_full_vs_inner_timeseries_with_diff(
    standard_gradients,
    inner_gradients,
    columns=[
        "x_gradient",
        "y_gradient",
        "z_gradient",
        "radial_gradient",
    ]
)

In [ ]:
plot_full_vs_inner_timeseries_with_diff(
    standard_bulk_stats,
    inner_bulk_stats,
    columns=[
        "mean_temp",
        "std_temp",
        "range_temp",
        "rate_per_30min",
    ]
)

***
We need to take a closer look at the fits, zooming in on each axis at a time. 

In [ ]:
def radialize_for_plot(temp, coords, fit, *, correct_radial_for_z=True):
    """
    For radial plots, optionally remove the radial-fit z term:
        T_corr = T - radial_z_gradient * z
    """
    if (
        correct_radial_for_z
        and "radial_z_gradient" in fit.index
        and np.all(np.isfinite(coords["z"]))
    ):
        return temp - fit["radial_z_gradient"] * coords["z"]

    return temp

def _resolve_timestamp(df, *, index=None, timestamp=None, random=False):
    if random:
        return np.random.choice(df.index)
    if index is not None:
        return df.index[index]
    if timestamp is not None:
        return timestamp
    raise ValueError("Provide one of: index=, timestamp=, or random=True")

In [ ]:
def plot_gradient_fit_inner_vs_full(
    thermocouples,
    *,
    index=None,
    timestamp=None,
    random=False,
    inner_radius_limit=m3_radius_limit,
    axes=("radial", "x", "y", "z"),
    full_gradients=None,
    inner_gradients=None,
    remove_nonstandard_cells=True,
    use_3d_dataset=True,
    correct_radial_for_z=True,
):
    """
    Plot gradient-fit diagnostics for Full M1M3 vs Inner M3.

    radial:
        optionally plots T - radial_z_gradient*z vs radius

    x/y/z:
        plots residualized temperatures with the other fitted terms removed.
    """

    if full_gradients is None:
        full_gradients = thermocouples.xyz_r_gradients

    if inner_gradients is None:
        inner_gradients = thermocouples.calculate_gradients_xyz_r(
            radius_limit=inner_radius_limit,
            remove_nonstandard_cells=remove_nonstandard_cells,
            use_3d_dataset=use_3d_dataset,
        )

    timestamp = _resolve_timestamp(
        full_gradients,
        index=index,
        timestamp=timestamp,
        random=random,
    )

    def get_data(radius_limit):
        xyz, temps = thermocouples._ThermocoupleAnalysis__coordinate_map(
            make_3d_map=use_3d_dataset,
            remove_nonstandard_cells=remove_nonstandard_cells,
            radius_limit=radius_limit,
        )

        if timestamp not in temps.index:
            ts = temps.index[
                temps.index.get_indexer([timestamp], method="nearest")[0]
            ]
        else:
            ts = timestamp

        row = temps.loc[ts]

        x = np.asarray(xyz[0]).astype(float)
        y = np.asarray(xyz[1]).astype(float)
        r = np.hypot(x, y)
        temp = row.to_numpy().astype(float)

        if use_3d_dataset:
            z = np.asarray(xyz[2]).astype(float)
        else:
            z = np.full_like(x, np.nan)

        coords = {
            "radial": r,
            "x": x,
            "y": y,
            "z": z,
        }

        return ts, coords, temp

    ts_full, coords_full, temp_full = get_data(radius_limit=None)
    ts_inner, coords_inner, temp_inner = get_data(
        radius_limit=inner_radius_limit
    )

    timestamp = ts_full

    full_fit = full_gradients.loc[timestamp]
    inner_fit = inner_gradients.loc[timestamp]

    gradient_col = {
        "radial": "radial_gradient",
        "x": "x_gradient",
        "y": "y_gradient",
        "z": "z_gradient",
    }

    xlabel = {
        "radial": "Radius (m)",
        "x": "X position (m)",
        "y": "Y position (m)",
        "z": "Z layer coordinate",
    }

    ylabel = {
        "radial": "Temperature (deg C)",
        "x": "Temperature corrected for Y/Z terms (deg C)",
        "y": "Temperature corrected for X/Z terms (deg C)",
        "z": "Temperature corrected for X/Y terms (deg C)",
    }

    title_name = {
        "radial": "Radial Gradient",
        "x": "X Gradient",
        "y": "Y Gradient",
        "z": "Z Gradient",
    }

    coord_gradient_cols = {
        "x": "x_gradient",
        "y": "y_gradient",
        "z": "z_gradient",
    }

    def residualize_for_axis(temp, coords, fit, axis):
        if axis == "radial":
            return temp

        corrected = temp.copy()

        for other_axis, other_col in coord_gradient_cols.items():
            if other_axis == axis:
                continue

            if other_col in fit.index:
                corrected = corrected - fit[other_col] * coords[other_axis]

        return corrected

    def correct_radial_temperature(temp, coords, fit):
        if not correct_radial_for_z:
            return temp

        if "radial_z_gradient" not in fit.index:
            print("WARNING: radial_z_gradient missing; radial data not z-corrected.")
            return temp

        if not np.any(np.isfinite(coords["z"])):
            print("WARNING: z coordinates unavailable; radial data not z-corrected.")
            return temp

        return temp - fit["radial_z_gradient"] * coords["z"]

    def get_radial_intercept(fit, label):
        if "radial_intercept" in fit.index:
            return fit["radial_intercept"]

        print(f"WARNING: {label} missing radial_intercept; using xyz intercept.")
        return fit["intercept"]

    for axis in axes:
        if axis not in gradient_col:
            raise ValueError(
                f"Unknown axis {axis!r}. Use one of {list(gradient_col)}."
            )

        if axis == "z" and not use_3d_dataset:
            print("Skipping z because use_3d_dataset=False.")
            continue

        col = gradient_col[axis]

        c_full = coords_full[axis]
        c_inner = coords_inner[axis]

        if axis == "radial":
            t_full = correct_radial_temperature(temp_full, coords_full, full_fit)
            t_inner = correct_radial_temperature(temp_inner, coords_inner, inner_fit)
        else:
            t_full = residualize_for_axis(temp_full, coords_full, full_fit, axis)
            t_inner = residualize_for_axis(temp_inner, coords_inner, inner_fit, axis)

        good_full = np.isfinite(c_full) & np.isfinite(t_full)
        good_inner = np.isfinite(c_inner) & np.isfinite(t_inner)

        c_full = c_full[good_full]
        t_full = t_full[good_full]

        c_inner = c_inner[good_inner]
        t_inner = t_inner[good_inner]

        c_grid_full = np.linspace(c_full.min(), c_full.max(), 200)
        c_grid_inner = np.linspace(c_inner.min(), c_inner.max(), 200)

        if axis == "radial":
            full_intercept = get_radial_intercept(full_fit, "Full M1M3")
            inner_intercept = get_radial_intercept(inner_fit, "Inner M3")

            t_fit_full = full_intercept + full_fit[col] * c_grid_full
            t_fit_inner = inner_intercept + inner_fit[col] * c_grid_inner
        else:
            t_fit_full = full_fit["intercept"] + full_fit[col] * c_grid_full
            t_fit_inner = inner_fit["intercept"] + inner_fit[col] * c_grid_inner

        fig, ax = plt.subplots(figsize=(9, 5))

        ax.scatter(
            c_full,
            t_full,
            color="0.7",
            alpha=0.55,
            s=25,
            label="Full M1M3 data",
        )

        ax.scatter(
            c_inner,
            t_inner,
            color="#0072B2",
            alpha=0.9,
            s=40,
            label=f"Inner M3 data (R < {inner_radius_limit:g} m)",
        )

        ax.plot(
            c_grid_full,
            t_fit_full,
            color="0.2",
            linewidth=2.5,
            label=f"Full M1M3 fit ({full_fit[col]:.4f} deg C/m)",
        )

        ax.plot(
            c_grid_inner,
            t_fit_inner,
            color="#0072B2",
            linestyle="--",
            linewidth=2.5,
            label=f"Inner M3 fit ({inner_fit[col]:.4f} deg C/m)",
        )

        if axis == "radial":
            ax.axvline(
                inner_radius_limit,
                color="0.4",
                linestyle=":",
                linewidth=1.5,
                label=f"Cut R={inner_radius_limit:g} m",
            )

        ax.set_xlabel(xlabel[axis])

        if (
            axis == "radial"
            and correct_radial_for_z
            and "radial_z_gradient" in full_fit.index
        ):
            ax.set_ylabel("Temperature corrected for radial-fit Z term (deg C)")
        else:
            ax.set_ylabel(ylabel[axis])

        ax.set_title(f"{title_name[axis]} Fit @ {timestamp}")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

    return timestamp

In [ ]:
ts = standard_gradients.index[0]
print(ts)

In [ ]:
plot_gradient_fit_inner_vs_full(
    thermocouples,
    timestamp=ts,
    inner_radius_limit=m3_radius_limit,
    correct_radial_for_z=True,
)